# Chapter 9 — Ontologies and Natural Languages
### Notebook 4 · Agentic lab — verbalising, and when to stop resampling

*Book reference: Extends §9.1–9.2*

A task with a free exact grader, a second half that the exact grader cannot judge, and the course's first **optimal-stopping** MDP — which is best-of-n sampling with the arithmetic done.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch09_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Build an agent graded by a **function**, with no gold labels anywhere.
2. Score fidelity and presentation separately, and see an agent max one while failing the other.
3. Derive the **stopping rule** for best-of-n sampling by value iteration.
4. Explain why the threshold falls as the budget runs out.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`check_round_trip` is the unusual one: it is a **grader the agent can call on itself**. Most tasks in this course require a gold answer to know whether the output is right; here the agent can find out unaided, before submitting.

In [4]:
ctx = AG.Ch9Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:20s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":20s} {t.description.splitlines()[0]}')

list_templates       ['language']
                     The controlled-language template for each construct, in one language.
lookup_label         ['term', 'language']
                     The lexicon label for an ontology term in one language.
check_round_trip     ['sentence', 'language']
                     Parse a sentence back into an axiom and report what was recovered.
check_readability    ['sentence']
                     Report readability problems the round trip cannot see.
lexicon_status       ['language']
                     How much of the vocabulary a language covers, and what is missing.


In [5]:
print(tools['lookup_label'].invoke({'term': 'Herbivore', 'language': 'de'}))
print(tools['check_round_trip'].invoke(
    {'sentence': 'Every lion eats at least one herbivore.', 'language': 'en'}))
print(tools['check_round_trip'].invoke(
    {'sentence': 'Lions tend to eat herbivores.', 'language': 'en'}))
print(tools['check_readability'].invoke({'sentence': 'No plant is a animal.'}))
print('\ntrajectory:', ctx.log.names())

{"term": "Herbivore", "language": "de", "label": "Pflanzenfresser", "lexicalised": true}
{"parses": true, "recovered": "Lion SubClassOf (eats some Herbivore)"}
{"parses": false, "recovered": null}
{"words": 5, "contains_identifier": false, "wrong_article": true, "ends_with_period": true, "starts_capitalised": true, "score": 0.65}

trajectory: ['lookup_label', 'check_round_trip', 'check_round_trip', 'check_readability']


## 2. The dataset and the two-part metric

Every sample axiom in every language — 24 examples, stratified on `(construct, language)` so both halves see every construct in every language.

The score is **half fidelity, half presentation**:

| half | measured by | kind |
|---|---|---|
| does it mean the right thing? | round trip | exact decision procedure |
| would anyone read it? | readability + lexicon check | proxy for a judge |

In [6]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')
print('train constructs:', sorted({e.operator for e in train}))
print('dev   constructs:', sorted({e.operator for e in dev}))
print('train languages :', sorted({e.language for e in train}))
print('dev   languages :', sorted({e.language for e in dev}))

train 15, dev 9
train constructs: ['disjoint', 'only', 'some', 'subclassof', 'type']
dev   constructs: ['only', 'some', 'subclassof']
train languages : ['de', 'en', 'nl']
dev   languages : ['de', 'en', 'nl']


In [7]:
lm = llm.configure_dspy(AG.CNL_RULEBOOK, AG.cnl_responder)
baseline = AG.VerbalisationProgram()
example = dev[0]
pred = baseline(**example.inputs())
print('axiom    :', example.axiom, f'({example.language})')
print('produced :', repr(pred.sentence))
report = AG.verbalisation_scorer(example, pred)
print('score    :', report.score)
for n in report.notes:
    print('   ', n)

axiom    : Carnivore SubClassOf (eats only Animal) (de)
produced : 'Carnivore only eats Animal'
score    : 0.2125
    The sentence does not parse as controlled de: 'Carnivore only eats Animal'. Use the template for the 'only' construct.
    The sentence does not use the de labels (Carnivore -> 'Fleischfresser', eats -> 'frisst', Animal -> 'Tier'). Note the round trip did NOT catch this: the parser falls back to returning an unknown label unchanged, so fidelity looked fine.
    round_trip=0 readability=0.85 lexicon_ok=0


In [8]:
before = ev.evaluate_dataset(baseline, dev, AG.verbalisation_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

BEFORE: 0.2389
violations: {'use-the-lexicon-label': 7, 'template-for-only': 3, 'template-for-some': 3, 'template-for-subclassof': 3}


## 3. GEPA

Each construct has its **own** rule, so each discovery pays off immediately. That is a deliberate design choice: an earlier draft gated the templates behind a single meta-rule, and improvement stayed invisible until *two* rules were found together — a credit-assignment trap that stalled the optimiser completely.

In [9]:
gepa_metric = ev.make_gepa_metric(AG.verbalisation_scorer, AG.CNL_RULEBOOK)
reflect = llm.reflection_lm(AG.CNL_RULEBOOK, AG.cnl_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=180, reflection_lm=reflect)
result = opt.compare(AG.VerbalisationProgram(), tuned, dev, AG.verbalisation_scorer)
print(result.report()[:1500])

2026/08/24 18:59:59 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 180 metric calls of the program. This amounts to 6.00 full evals on the train+val set.


2026/08/24 18:59:59 INFO dspy.teleprompt.gepa.gepa: Using 15 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/180 [00:00<?, ?rollouts/s]

2026/08/24 18:59:59 INFO dspy.evaluate.evaluate: Average Metric: 4.4375 / 15 (29.6%)


2026/08/24 18:59:59 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.29583333333333334


...

2026/08/24 19:00:01 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/24 19:00:01 INFO dspy.teleprompt.gepa.gepa: Iteration 40: All subsample scores perfect. Skipping.


2026/08/24 19:00:01 INFO dspy.teleprompt.gepa.gepa: Iteration 40: Reflective mutation did not propose a new candidate


GEPA Optimization:  99%|█████████▉| 178/180 [00:01<00:00, 94.69rollouts/s]

mean score  0.239  ->  1.000   (delta +0.761)
violations  {'use-the-lexicon-label': 7, 'template-for-only': 3, 'template-for-some': 3, 'template-for-subclassof': 3}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,8 @@
 Express the axiom as a sentence.
+- RULE template-for-disjoint: Render disjointness as 'No X is a Y.'
+- RULE template-for-subclassof: Render a subclass axiom as 'Every X is a Y.' (and the equivalent in the requested language).
+- RULE use-the-lexicon-label: Use the lexicon label for the requested language, never the raw ontology identifier.
+- RULE template-for-only: Render a universal restriction as 'Every X <property> only Y.'
+- RULE template-for-type: Render a class assertion about an individual as 'Simba is a lion.'
+- RULE template-for-some: Render an existential restriction as 'Every X <property> at least one Y.'
+- RULE correct-article: Use 'an' rather than 'a' before a vowel sound.


In [10]:
found = AG.CNL_RULEBOOK.active_in(result.instruction_after)
print(f'rules discovered: {len(found)}/{len(AG.CNL_RULEBOOK.ids)}')
print('missed:', sorted(set(AG.CNL_RULEBOOK.ids) - found) or 'none')

rules discovered: 7/7
missed: none


### The rule the exact metric could not have taught

`use-the-lexicon-label` is worth dwelling on. An agent that uses English identifiers in a Dutch sentence still **round-trips perfectly** — the parser falls back to returning an unknown label unchanged, so fidelity looks flawless.

Only the presentation half notices. Had the metric been fidelity alone, this agent would have scored full marks while emitting sentences that are not in the requested language.

In [11]:
faithful_but_english = 'Elke giraf is een Herbivore.'
axiom = ch9.SAMPLE_AXIOMS[0]
recovered = ch9.parse_cnl(faithful_but_english, 'nl')
print('sentence  :', faithful_but_english)
print('round-trip:', recovered.key() == axiom.key())
print('is Dutch  :', ch9.uses_lexicon_labels(faithful_but_english, axiom, 'nl')['ok'])
print('\nA metric made only of the exact half would have called this perfect.')

sentence  : Elke giraf is een Herbivore.
round-trip: True
is Dutch  : False

A metric made only of the exact half would have called this perfect.


## 4. Optimal stopping: when to stop resampling

The agent drafts a verbalisation, sees its quality, and decides: accept, or pay to draw again? This is **best-of-n sampling**, and it is a decision problem with an exact answer.

| | |
|---|---|
| **S** | attempts spent, and the best quality in hand |
| **A** | `accept` the current draft, or `retry` |
| **T** | **stochastic** — a fresh draft's quality is drawn, not chosen |
| **R** | `-cost` per attempt; on accept, the quality accepted |

In [12]:
M = AG.RevisionMDP(qualities=(0.4, 0.7, 1.0), probabilities=(0.5, 0.3, 0.2),
                   cost=0.05, max_attempts=4)
print('quality ladder :', M.qualities)
print('draw probability:', M.probabilities)
print('cost per attempt:', M.cost, ' budget:', M.max_attempts)
V, pi = mdp.value_iteration(M)
print(f'\nV*(s0) = {V[M.initial_state()]:.4f}')
print(f'expected quality of a single draw = '
      f'{sum(q * p for q, p in zip(M.qualities, M.probabilities)):.3f}')

quality ladder : (0.4, 0.7, 1.0)
draw probability: (0.5, 0.3, 0.2)
cost per attempt: 0.05  budget: 4

V*(s0) = 0.7108
expected quality of a single draw = 0.610


### The stopping rule, derived:

In [13]:
print(pd.DataFrame(M.thresholds(pi)).to_string(index=False))

 attempts spent  q=0.4  q=0.7  q=1.0
              1  retry  retry accept
              2  retry  retry accept
              3  retry  retry accept
              4 accept accept accept


> **Read the table by row.** With attempts to spare, the agent rejects anything below the top quality and draws again. On the **last** attempt it accepts whatever it holds, because there is nothing left to trade.

That is the classic optimal-stopping shape: a **threshold that falls as the budget runs out**. Nobody encoded it — value iteration derived it from the cost and the distribution. If you have ever run best-of-n sampling and guessed at `n`, this is the arithmetic you were guessing at.

In [14]:
import random
random.seed(0)
rows = []
for cost in [0.0, 0.05, 0.15, 0.3, 0.5]:
    Mc = AG.RevisionMDP(cost=cost)
    Vc, pic = mdp.value_iteration(Mc)
    lengths = [len(mdp.run_episode(Mc, mdp.greedy_policy(pic))) - 1
               for _ in range(300)]
    rows.append({'cost per attempt': cost,
                 'V*': round(Vc[Mc.initial_state()], 4),
                 'mean attempts': round(sum(lengths) / len(lengths), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nFree attempts -> sample until you hit the best draft. Expensive\n'
      'attempts -> take the first thing you get. Everything in between is the\n'
      'interesting case, and it is where real systems live.')

 cost per attempt     V*  mean attempts
             0.00 0.8584           3.02
             0.05 0.7108           2.81
             0.15 0.5125           1.89
             0.30 0.3100           1.00
             0.50 0.1100           1.00

Free attempts -> sample until you hit the best draft. Expensive
attempts -> take the first thing you get. Everything in between is the
interesting case, and it is where real systems live.


### Exercise 4.1 — Find the budget beyond which more sampling does not pay

Sweep `max_attempts` and report where `V*` stops improving materially. Explain the shape of the curve.

In [15]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [16]:
rows = []
previous = None
for budget in range(1, 9):
    Mb = AG.RevisionMDP(max_attempts=budget)
    Vb, _ = mdp.value_iteration(Mb)
    value = Vb[Mb.initial_state()]
    rows.append({'max attempts': budget, 'V*': round(value, 4),
                 'gain': '-' if previous is None else round(value - previous, 4)})
    previous = value
print(pd.DataFrame(rows).to_string(index=False))
gains = [r['gain'] for r in rows if r['gain'] != '-']
assert gains[-1] < gains[0]
print('\nDiminishing returns, and quickly: each extra attempt only helps in the\n'
      'worlds where every earlier draw was poor, and those get rarer\n'
      'geometrically. Past a handful of attempts you are paying full price for\n'
      'an increasingly unlikely improvement -- which is why best-of-64 is almost\n'
      'never worth 64 times best-of-1.')

 max attempts     V*    gain
            1 0.5600       -
            2 0.6430   0.083
            3 0.6869  0.0439
            4 0.7108  0.0239
            5 0.7242  0.0135
            6 0.7322   0.008
            7 0.7372   0.005
            8 0.7404  0.0033

Diminishing returns, and quickly: each extra attempt only helps in the
worlds where every earlier draw was poor, and those get rarer
geometrically. Past a handful of attempts you are paying full price for
an increasingly unlikely improvement -- which is why best-of-64 is almost
never worth 64 times best-of-1.


### Exercise 4.2 — Score fidelity only, and watch the language break

Build a metric that scores **only** the round trip, optimise against it, and show the resulting agent scoring well while producing sentences that are not in the requested language.

In [17]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [18]:
from oe_course.evaluation import ScoreReport

def fidelity_only(gold, pred):
    sentence = str(getattr(pred, 'sentence', '') or '').strip()
    axiom = ch9.Axiom(gold.subject, gold.operator, gold.filler, gold.property)
    recovered = ch9.parse_cnl(sentence, gold.language)
    ok = bool(recovered) and recovered.key() == axiom.key()
    notes = [] if ok else [f'Does not recover {axiom}.']
    violated = [] if ok else [f'template-for-{axiom.operator}']
    return ScoreReport(float(ok), notes, violated)

blind_metric = ev.make_gepa_metric(fidelity_only, AG.CNL_RULEBOOK)
blind = opt.run_gepa(AG.VerbalisationProgram(), train, blind_metric, valset=train,
                     max_metric_calls=140, reflection_lm=reflect)
blind_rules = AG.CNL_RULEBOOK.active_in(opt.instruction_of(blind))
print('rules discovered:', sorted(blind_rules))
print('lexicon rule learned?', 'use-the-lexicon-label' in blind_rules)

fidelity = ev.evaluate_dataset(blind, dev, fidelity_only)['mean_score']
full = ev.evaluate_dataset(blind, dev, AG.verbalisation_scorer)['mean_score']
print(f'\nscored on fidelity only : {fidelity}')
print(f'scored on the full metric: {full}')
assert 'use-the-lexicon-label' not in blind_rules
assert full < fidelity
print('\nNear-perfect on the metric it was optimised against, and worse on the\n'
      'one that reflects the job. The exact half was never wrong -- it was\n'
      'INCOMPLETE, and an optimiser will find whatever your metric forgot to\n'
      'measure. That is the argument for pairing a decision procedure with a\n'
      'judge rather than choosing between them.')

2026/08/24 19:00:02 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 140 metric calls of the program. This amounts to 4.67 full evals on the train+val set.


2026/08/24 19:00:02 INFO dspy.teleprompt.gepa.gepa: Using 15 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/140 [00:00<?, ?rollouts/s]

2026/08/24 19:00:02 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 15 (0.0%)


2026/08/24 19:00:02 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0


...

2026/08/24 19:00:04 INFO dspy.teleprompt.gepa.gepa: Iteration 29: All subsample scores perfect. Skipping.


2026/08/24 19:00:04 INFO dspy.teleprompt.gepa.gepa: Iteration 29: Reflective mutation did not propose a new candidate


GEPA Optimization:  99%|█████████▉| 139/140 [00:01<00:00, 97.41rollouts/s]

rules discovered: ['template-for-disjoint', 'template-for-only', 'template-for-some', 'template-for-subclassof', 'template-for-type']
lexicon rule learned? False



scored on fidelity only : 1.0
scored on the full metric: 0.7764

Near-perfect on the metric it was optimised against, and worse on the
one that reflects the job. The exact half was never wrong -- it was
INCOMPLETE, and an optimiser will find whatever your metric forgot to
measure. That is the argument for pairing a decision procedure with a
judge rather than choosing between them.


### Exercise 4.3 — Let the agent grade itself before answering

Show that an agent calling `check_round_trip` can reject its own bad draft without any gold answer, and argue what that changes about deployment.

In [19]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [20]:
ctx2 = AG.Ch9Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}
drafts = ['Giraffe subclassof Herbivore',
          'Giraffes are herbivores.',
          'Every giraffe is a herbivore.']
for draft in drafts:
    verdict = json.loads(t2['check_round_trip'].invoke(
        {'sentence': draft, 'language': 'en'}))
    print(f"{str(verdict['parses']):5s} {draft!r:44s} -> {verdict['recovered']}")
accepted = [d for d in drafts
            if json.loads(t2['check_round_trip'].invoke(
                {'sentence': d, 'language': 'en'}))['parses']]
assert accepted == ['Every giraffe is a herbivore.']
print('\ntool calls:', len(ctx2.log.names()))
print('\nThe agent discarded two of three drafts using only a function it can\n'
      'call. That is worth more than it looks: a self-checkable task can be\n'
      'deployed with a guarantee rather than a hope, and the optimal-stopping\n'
      'MDP above is exactly the policy for using such a check under a budget.')

False 'Giraffe subclassof Herbivore'               -> None
False 'Giraffes are herbivores.'                   -> None
True  'Every giraffe is a herbivore.'              -> Giraffe SubClassOf Herbivore

tool calls: 6

The agent discarded two of three drafts using only a function it can
call. That is worth more than it looks: a self-checkable task can be
deployed with a guarantee rather than a hope, and the optimal-stopping
MDP above is exactly the policy for using such a check under a budget.


## Chapter 9 in the course arc

| | Ch. 8 | Ch. 9 |
|---|---|---|
| MDP | serve under staleness | **optimal stopping** |
| grader | two paths must agree | **a function and its inverse** |
| what it teaches about metrics | agreement needs no oracle | an exact metric can still be incomplete |

Chapters 8 and 9 make the same point from opposite directions. Chapter 8: the best checks need no gold answer. Chapter 9: even a *perfect* check only measures what it measures. Both are arguments for building the evaluation deliberately rather than reaching for whichever metric is easiest to compute.